# 00 — Setup

Bootstraps everything the rest of the demo needs:

1. **Spark / Databricks Connect** session is live (or you're inside a Databricks workspace notebook)
2. **Unity Catalog** target schema exists at `{UC_CATALOG}.{UC_SCHEMA}`
3. **MLflow** is pointed at the UC model registry and the experiment exists

Run this first. Every other notebook expects the schema + experiment.

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

# Dual-mode Spark — local via Databricks Connect, or workspace where `spark` exists
try:
    spark
except NameError:
    from databricks.connect import DatabricksSession
    spark = DatabricksSession.builder.serverless().getOrCreate()

from databricks.sdk import WorkspaceClient
w = WorkspaceClient()
me = w.current_user.me().user_name
print(f"Authenticated as: {me}")

/Users/alexander.booth/miniconda3/envs/demo-env/lib/python3.12/site-packages/pyspark/sql/connect/client/core.py:141: UserWarning: Could not enable SetAllowOversizeProtos, please check the protobuf version.
  warnings.warn("Could not enable SetAllowOversizeProtos, please check the protobuf version.")


Authenticated as: alexander.booth@databricks.com


In [2]:
UC_CATALOG  = os.getenv("UC_CATALOG", "alexander_booth")
UC_SCHEMA   = os.getenv("UC_SCHEMA", "hockey_xg_mlflow")
EXPERIMENT  = os.getenv("MLFLOW_EXPERIMENT_NAME", f"/Users/{me}/hockey-xg-mlflow")
MODEL_NAME  = f"{UC_CATALOG}.{UC_SCHEMA}.xg_model"
ENDPOINT    = os.getenv("SERVING_ENDPOINT_NAME", "hockey-xg-endpoint")

print(f"Catalog:    {UC_CATALOG}")
print(f"Schema:     {UC_CATALOG}.{UC_SCHEMA}")
print(f"Model:      {MODEL_NAME}")
print(f"Experiment: {EXPERIMENT}")
print(f"Endpoint:   {ENDPOINT}")

Catalog:    alexander_booth
Schema:     alexander_booth.hockey_xg_mlflow
Model:      alexander_booth.hockey_xg_mlflow.xg_model
Experiment: /Users/alexander.booth@databricks.com/hockey-xg-mlflow
Endpoint:   hockey-xg-endpoint


In [3]:
spark.sql("SELECT current_user(), current_timestamp(), current_catalog()").show(truncate=False)
print("Spark version:", spark.version)

+------------------------------+--------------------------+-----------------+
|current_user()                |current_timestamp()       |current_catalog()|
+------------------------------+--------------------------+-----------------+
|alexander.booth@databricks.com|2026-05-29 03:18:11.037794|main             |
+------------------------------+--------------------------+-----------------+

Spark version: 4.1.0


In [4]:
# Create UC catalog (if user has perms) + schema. Idempotent.
try:
    spark.sql(f"CREATE CATALOG IF NOT EXISTS {UC_CATALOG}")
except Exception as e:
    print(f"  catalog create skipped: {e}")

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {UC_CATALOG}.{UC_SCHEMA}")
print(f"  schema ready: {UC_CATALOG}.{UC_SCHEMA}")

  schema ready: alexander_booth.hockey_xg_mlflow


## MLflow — point at UC model registry + create experiment

On Databricks the registry should be **UC** (not the legacy workspace registry). UC
registry gives you catalog-level governance, lineage, and aliases per model version.

In [ ]:
import mlflow

mlflow.set_tracking_uri("databricks")
mlflow.set_registry_uri("databricks-uc")
exp = mlflow.set_experiment(EXPERIMENT)
print(f"Experiment ID:   {exp.experiment_id}")
print(f"Experiment path: {exp.name}")
print(f"Registry URI:    {mlflow.get_registry_uri()}")

## Sanity check

If everything above ran without error, you're good to move to `01_generate_synthetic_shots`.
If you saw a permissions error on `CREATE CATALOG`, ask a workspace admin to grant
`CREATE SCHEMA` on the catalog and re-run — the rest of the demo only needs schema-level perms.